In [2]:
from molsim import MolecularDynamics
import numpy as np
import matplotlib.pyplot as plt
import time

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

# Exercise 4: Molecular Dynamics Optimization

## Question 1

In this exercise, we are going to look at scaling: how much more expensive is the simulations as a function of the number of particles. You can use jupyters timing function `%%timeit` to benchmark your implementations. This function will run the code 7 times and report the average time it took to complete. 

In [13]:
# Set up the simulation
config = {
    "numberOfParticles": 200,
    "temperature": 1.0,
    "dt": 0.005,
    "boxSize": np.cbrt(16.0) * 8.0,
    "logLevel": 0,
    "seed": 12,
    "numberOfEquilibrationSteps": 20000,
    "numberOfProductionSteps": 20000,
    "sampleFrequency": 100,
}

In [ ]:
%%timeit
# Run the simulation and time it
md = MolecularDynamics(**config)
md.run()

In [ ]:
# start refactor
npart = []
times = []
# end refactor

# Plot the results
fig, ax = plt.subplots()
ax.set_xscale("log")
ax.set_yscale("log")
ax.scatter(npart, times, c="red", marker="o", label="Optimized implementation")
ax.set_xlabel("Number of particles")
ax.set_ylabel("Execution time / s")

n = np.linspace(min(npart), max(npart), 100)
ax.plot(n, (0.01 * n) ** 2, label=r"$\mathcal{O}(N^2)$", c="black", ls="--")
ax.set_xlim([1e1, 1e3])
ax.legend(fontsize=14)

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

## Question 2

Most MD package use tricks to optimize the code. When optimizing code it is best to focus on the parts that are executed most often. In our case, the energy and force calculation are the most computationally demanding. 

Have a look at the force computation in:
`src/molecularDynamics/md.cpp`
in function:
`MolecularDynamics::calculateForce`

##### Lower triangular force computation
The double loop $i, j$ over the numbers of particles both loop over N, while we only need to calculate the lower triangular part of the matrix. Another reason why this is inefficient is because we need an if statement checking $i!=j$ in the inner loop. If statements make code unpredictable for the computer, leading to reduced optimization. 

We can change the loop from $i (0, ... N)$ and $j (0, ... N)$ by only considering pairs where $j$ is larger than $i$. The easiest option is replacing `(i != j)` with `(i < j)`, but this does not remove the if statement. Another way would be to loop $i$ from $(0, ... N - 1)$ and $j$ from $(i+1, ... N)$. However, we should consider that the a *force* added to $i$, should cause a counter force added to $j$. 

##### Removing pow
Raising powers is expensive for computers! While very architecture dependent here is an overview of clock ticks per operation on floating points:
- Addition / subtraction: 3 cycles
- Multiplication: 3 cycles
- Division: 10-20 cycles
- Square root: 10-20 cycles
- pow, exp, sin, cos, log: 20-100+ cycles

This is why it can be beneficial to prevent the amount of uses of the pow function. In this case we can easily get rid of the std::sqrt function, by working with $r^2$. Furthermore, we can get rid std::pow(r, -12) function by computing $r^{-2}$ and then $r^{-6}$. Precomputing $r^{-6}$ costs one division and 3 multiplications.

In this exercise, measure the difference between two implementations of the force-loop. In the first implementation, use `MolecularDynamics::calculateForceSlow`, then use `MolecularDynamics::calculateForceFast`. Compare the difference in speed.
You can use jupyters timing function `%%timeit` to benchmark your implementations. This function will run the code 7 times and report the average time it took to complete. 